In [0]:

# Production Databricks Transaction Processing Pipeline
# Environment-aware, auditable, idempotent processing

#============================================
# 1. PARAMETERS + IMPORTS
# ============================================

from pyspark.sql import functions as F
from datetime import datetime
import uuid

# Job parameters
dbutils.widgets.text("environment", "DEV")
dbutils.widgets.text("process_date", "2026-08-16")
dbutils.widgets.text("batch_id", "BATCH_001")

environment = dbutils.widgets.get("environment")
process_date = dbutils.widgets.get("process_date")
batch_id = dbutils.widgets.get("batch_id")

print("Environment :", environment)
print("Process Date:", process_date)
print("Batch ID    :", batch_id)

In [0]:
# ============================================
# 2. VALIDATION + RUN ID
# ============================================

allowed_environments = {"DEV", "TEST", "PROD"}

if environment not in allowed_environments:
    raise ValueError(
        f"Invalid environment: {environment}"
    )

try:
    datetime.strptime(process_date, "%Y-%m-%d")
except ValueError:
    raise ValueError(
        f"Invalid process_date: {process_date}"
    )

if not batch_id.strip():
    raise ValueError("batch_id cannot be empty")

run_id = str(uuid.uuid4())
start_time = datetime.now()

print("Parameter validation: PASSED")
print("Run ID:", run_id)
print("Start :", start_time)

In [0]:
# ============================================
# 3. READ BRONZE + BUILD DATASETS
# ============================================

bronze_table = "banking_cat.prod_banking.bronze_transactions"

df_bronze = spark.table(bronze_table)

source_records = df_bronze.count()

# Silver
df_silver = (
    df_bronze
    .filter(F.col("status") == "SUCCESS")
    .filter(F.col("amount") > 0)
    .withColumn("processing_batch_id", F.lit(batch_id))
    .withColumn("processing_environment", F.lit(environment))
    .withColumn("processed_timestamp", F.current_timestamp())
)

valid_records = df_silver.count()

# Quarantine
df_quarantine = (
    df_bronze
    .filter(
        (F.col("status") != "SUCCESS") |
        (F.col("amount") <= 0)
    )
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("status") != "SUCCESS",
            F.lit("TRANSACTION_STATUS_NOT_SUCCESS")
        )
        .when(
            F.col("amount") <= 0,
            F.lit("INVALID_AMOUNT")
        )
    )
    .withColumn("processing_batch_id", F.lit(batch_id))
    .withColumn("processing_environment", F.lit(environment))
    .withColumn("rejected_timestamp", F.current_timestamp())
)

rejected_records = df_quarantine.count()

# Gold
df_gold = (
    df_silver
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_transaction_amount")
    )
    .withColumn("processing_batch_id", F.lit(batch_id))
    .withColumn("processing_environment", F.lit(environment))
    .withColumn("processed_timestamp", F.current_timestamp())
)

gold_records = df_gold.count()

print("Source records    :", source_records)
print("Valid records     :", valid_records)
print("Rejected records  :", rejected_records)
print("Gold records      :", gold_records)

In [0]:
# ============================================
# 4. RECONCILIATION + DELTA MERGE
# ============================================

# Reconciliation
if source_records != valid_records + rejected_records:
    raise ValueError(
        f"Reconciliation failed: "
        f"{source_records} != "
        f"{valid_records} + {rejected_records}"
    )

print("Reconciliation: PASSED")


# Temporary views
df_silver.createOrReplaceTempView("silver_source")
df_quarantine.createOrReplaceTempView("quarantine_source")
df_gold.createOrReplaceTempView("gold_source")


# Silver MERGE
spark.sql("""
MERGE INTO banking_cat.prod_banking.silver_transactions AS target
USING silver_source AS source
ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")


# Quarantine MERGE
spark.sql("""
MERGE INTO banking_cat.prod_banking.quarantine_transactions AS target
USING quarantine_source AS source
ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")


# Gold MERGE
spark.sql("""
MERGE INTO banking_cat.prod_banking.gold_customer_transactions AS target
USING gold_source AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("Silver MERGE       : completed")
print("Quarantine MERGE   : completed")
print("Gold MERGE         : completed")

In [0]:
# ============================================
# AUDIT FUNCTION
# ============================================

def write_audit_success(
    batch_id,
    run_id,
    job_name,
    environment,
    process_date,
    start_time,
    end_time,
    source_records,
    valid_records,
    rejected_records,
    gold_records
):
    duration_seconds = (
        end_time - start_time
    ).total_seconds()

    spark.sql(f"""
    INSERT INTO banking_cat.prod_banking.job_audit
    (
        batch_id,
        job_name,
        environment,
        process_date,
        start_time,
        end_time,
        status,
        records_processed,
        error_message,
        source_records,
        valid_records,
        rejected_records,
        gold_records,
        run_id,
        execution_duration_seconds
    )
    VALUES
    (
        '{batch_id}',
        '{job_name}',
        '{environment}',
        DATE('{process_date}'),
        TIMESTAMP('{start_time}'),
        TIMESTAMP('{end_time}'),
        'SUCCESS',
        {source_records},
        NULL,
        {source_records},
        {valid_records},
        {rejected_records},
        {gold_records},
        '{run_id}',
        {duration_seconds}
    )
    """)

print("Audit function loaded")

In [0]:
# ============================================
# 5. FINAL VALIDATION + SUCCESS AUDIT
# ============================================

# Final table counts
silver_count = spark.table(
    "banking_cat.prod_banking.silver_transactions"
).count()

quarantine_count = spark.table(
    "banking_cat.prod_banking.quarantine_transactions"
).count()

gold_count = spark.table(
    "banking_cat.prod_banking.gold_customer_transactions"
).count()

# Basic production validation
if silver_count < valid_records:
    raise ValueError("Silver validation failed")

if quarantine_count < rejected_records:
    raise ValueError("Quarantine validation failed")

if gold_count < gold_records:
    raise ValueError("Gold validation failed")

end_time = datetime.now()

# Write SUCCESS audit
write_audit_success(
    batch_id=batch_id,
    run_id=run_id,
    job_name="PROD_TXN_PROCESSING_JOB",
    environment=environment,
    process_date=process_date,
    start_time=start_time,
    end_time=end_time,
    source_records=source_records,
    valid_records=valid_records,
    rejected_records=rejected_records,
    gold_records=gold_records
)

print("===================================")
print("PIPELINE SUCCESS")
print("===================================")
print("Run ID       :", run_id)
print("Silver       :", silver_count)
print("Quarantine   :", quarantine_count)
print("Gold         :", gold_count)
print("Duration     :", round(
    (end_time - start_time).total_seconds(), 2
), "seconds")

In [0]:
# ============================================
# 6. ERROR HANDLING
# ============================================

def write_audit_failure(
    batch_id,
    run_id,
    job_name,
    environment,
    process_date,
    start_time,
    end_time,   
    error_message
):
    duration_seconds = (
        end_time - start_time
    ).total_seconds()

    safe_error = str(error_message).replace("'", "''")

    spark.sql(f"""
    INSERT INTO banking_cat.prod_banking.job_audit
    (
        batch_id,
        job_name,
        environment,
        process_date,
        start_time,
        end_time,
        status,
        records_processed,
        error_message,
        run_id,
        execution_duration_seconds
    )
    VALUES
    (
        '{batch_id}',
        '{job_name}',
        '{environment}',
        DATE('{process_date}'),
        TIMESTAMP('{start_time}'),
        TIMESTAMP('{end_time}'),
        'FAILED',
        0,
        '{safe_error}',
        '{run_id}',
        {duration_seconds}
    )
    """)

print("Failure audit function loaded")

In [0]:
# ============================================
# 7. FINAL NOTEBOOK VALIDATION
# ============================================

print("===================================")
print("PRODUCTION NOTEBOOK VALIDATION")
print("===================================")

print("Environment :", environment)
print("Process Date:", process_date)
print("Batch ID    :", batch_id)
print("Run ID      :", run_id)

print("-----------------------------------")

print(
    "Silver records:",
    spark.table(
        "banking_cat.prod_banking.silver_transactions"
    ).count()
)

print(
    "Quarantine records:",
    spark.table(
        "banking_cat.prod_banking.quarantine_transactions"
    ).count()
)

print(
    "Gold records:",
    spark.table(
        "banking_cat.prod_banking.gold_customer_transactions"
    ).count()
)

print("-----------------------------------")
print("Notebook validation: PASSED")

In [0]:
spark.sql("""
SELECT
    run_id,
    batch_id,
    environment,
    status,
    source_records,
    valid_records,
    rejected_records,
    gold_records,
    execution_duration_seconds
FROM banking_cat.prod_banking.job_audit
ORDER BY start_time DESC
LIMIT 5
""").show(truncate=False)